In [6]:
import pandas as pd

# Ruta a tus archivos
ipc_path = "17_1_04_IPC_variaciones_interanuales_segun_divisiones_de_la_canasta_categorias_bienes_y_servicios._Patagonia.xlsx"
ingresos_path = "21_1_01_Remuneracion-promedio-de-los-trabajadores-registrados-del-sector-privado-segun-rama-de-actividad.xlsx"
tarjetas_path = "Series_anual_2023_20240507.xlsx"

# ---------------------------
# 1. Procesar datos de IPC
# ---------------------------
ipc = pd.read_excel(ipc_path, sheet_name="Nivel general", skiprows=5)
ipc = ipc[ipc["Categoría"].notna()]
ipc = ipc[["Categoría"] + [col for col in ipc.columns if isinstance(col, pd.Timestamp)]]
ipc = ipc[ipc["Categoría"] == "Nivel general"]

ipc_t = ipc.set_index("Categoría").T.reset_index()
ipc_t.columns = ["Fecha", "IPC_Interanual"]
ipc_t["Fecha"] = pd.to_datetime(ipc_t["Fecha"], format="%Y-%m", errors="coerce")
ipc_t = ipc_t.dropna(subset=["Fecha"])
ipc_t = ipc_t[ipc_t["Fecha"].dt.year >= 2017]

# ---------------------------
# 2. Procesar datos de ingresos
# ---------------------------
ingresos = pd.read_excel(ingresos_path, sheet_name=1, header=1)
ingresos = ingresos.rename(columns={"Unnamed: 0": "Fecha"})
ingresos["Fecha"] = pd.to_datetime(ingresos["Fecha"], format="%Y-%m", errors="coerce")
ingresos = ingresos.dropna(subset=["Fecha"])

# Promedio general del total de actividades
ingresos["Ingreso_Promedio"] = ingresos.drop(columns="Fecha").mean(axis=1)
ingresos_t = ingresos[["Fecha", "Ingreso_Promedio"]]
ingresos_t = ingresos_t[ingresos_t["Fecha"].dt.year >= 2017]

# ---------------------------
# 3. Procesar datos de tarjetas
# ---------------------------
tarjetas = pd.read_excel(tarjetas_path, sheet_name="Pagos minoristas")

# Renombrar columnas específicas
tarjetas = tarjetas.rename(columns={
    "Tarjetas de crédito": "Tarjeta_Credito_Tipo",
    "Unnamed: 35": "Tarjeta_Credito_Monto",
    "Unnamed: 36": "Tarjeta_Credito_Operaciones"
})

tarjetas["Fecha"] = pd.to_datetime(tarjetas["Fecha"], errors="coerce")
tarjetas = tarjetas[tarjetas["Fecha"].dt.year >= 2017]
tarjetas = tarjetas.dropna(subset=["Tarjeta_Credito_Monto", "Tarjeta_Credito_Operaciones"])

# ---------------------------
# 4. Unir todo en un solo DataFrame
# ---------------------------
df = pd.merge(ipc_t, ingresos_t, on="Fecha", how="inner")
df = pd.merge(df, tarjetas[["Fecha", "Tarjeta_Credito_Monto", "Tarjeta_Credito_Operaciones"]], on="Fecha", how="left")

# ---------------------------
# 5. Exportar a CSV
# ---------------------------
df.to_csv("datos_unificados_2017_2023.csv", index=False)

print("✅ Datos procesados y exportados correctamente como 'datos_unificados_2017_2023.csv'")


ValueError: Worksheet named 'Nivel general' not found